In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg , udf
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (101, "Aarav", "HR", 45000),
    (102, "Diya", "IT", 60000),
    (103, "Vivaan", "Finance", 55000),
    (104, "Anaya", "Marketing", 50000),
    (105, "Aditya", "IT", 65000),
    (106, "Ishita", "HR", 47000),
    (107, "Krishna", "Sales", 52000),
    (108, "Meera", "Finance", 58000),
    (109, "Arjun", "IT", 70000),
    (110, "Riya", "Marketing", 49000),
    (111, "Kabir", "Sales", 54000),
    (112, "Saanvi", "HR", 46000),
    (113, "Reyansh", "IT", 72000),
    (114, "Aadhya", "Finance", 61000),
    (115, "Atharv", "Sales", 53000),
    (116, "Myra", "Marketing", 51000),
    (117, "Shaurya", "IT", 68000),
    (118, "Pari", "HR", 48000),
    (119, "Yash", "Finance", 59000),
    (120, "Kiara", "Sales", 56000)
]
columns = ["Id", "Name", "Department", "Salary"]
df = spark.createDataFrame(data, columns);

In [2]:
df.write.mode("overwrite").csv("output/employee_data",header = True)

In [3]:
df.toPandas().to_csv(
    "output/employeetable2.csv",
    index = False
)

In [4]:
df.take(10)

[Row(Id=101, Name='Aarav', Department='HR', Salary=45000),
 Row(Id=102, Name='Diya', Department='IT', Salary=60000),
 Row(Id=103, Name='Vivaan', Department='Finance', Salary=55000),
 Row(Id=104, Name='Anaya', Department='Marketing', Salary=50000),
 Row(Id=105, Name='Aditya', Department='IT', Salary=65000),
 Row(Id=106, Name='Ishita', Department='HR', Salary=47000),
 Row(Id=107, Name='Krishna', Department='Sales', Salary=52000),
 Row(Id=108, Name='Meera', Department='Finance', Salary=58000),
 Row(Id=109, Name='Arjun', Department='IT', Salary=70000),
 Row(Id=110, Name='Riya', Department='Marketing', Salary=49000)]

In [6]:
print("before partitionings:",df.rdd.getNumPartitions())

before partitionings: 12


In [7]:
df_repartition = df.repartition(5)
print("After partition:",df_repartition.rdd.getNumPartitions())

[Stage 5:>                                                        (0 + 12) / 12]

After partition: 5


[Stage 5:====>                                                    (1 + 11) / 12]

In [8]:
df_colesced = df_repartition.coalesce(2)
print("after coalesced:",df_colesced.rdd.getNumPartitions())

after coalesced: 2


In [9]:
df_repartition.write.mode("overwrite").csv("output/employee_data_afterP",header = True)

In [10]:
optimized_df = df.filter(col("Salary")>55000).filter(col("Salary")<70000)
optimized_df.show()

+---+-------+----------+------+
| Id|   Name|Department|Salary|
+---+-------+----------+------+
|102|   Diya|        IT| 60000|
|105| Aditya|        IT| 65000|
|108|  Meera|   Finance| 58000|
|114| Aadhya|   Finance| 61000|
|117|Shaurya|        IT| 68000|
|119|   Yash|   Finance| 59000|
|120|  Kiara|     Sales| 56000|
+---+-------+----------+------+



In [11]:
optimized_df.explain()

== Physical Plan ==
*(1) Filter (isnotnull(Salary#3L) AND ((Salary#3L > 55000) AND (Salary#3L < 70000)))
+- *(1) Scan ExistingRDD[Id#0L,Name#1,Department#2,Salary#3L]




In [16]:
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. Optimized execution | count: {count_uncached} | time: {round(end_time - start_time, 4)} seconds")

[Stage 16:>                                                       (0 + 12) / 12]

1. Optimized execution | count: 7 | time: 1.0804 seconds


In [17]:
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. Optimized execution | count: {count_uncached} | time: {round(end_time - start_time, 4)} seconds")

1. Optimized execution | count: 7 | time: 0.4422 seconds


In [12]:
import time
start_time = time.time()
count_first_cache = optimized_df.count()
end_time = time.time()
print(f"1. Optimized execution | count: {count_first_cache} | time: {round(end_time - start_time, 2)} seconds")

1. Optimized execution | count: 7 | time: 0.71 seconds


In [15]:
optimized_df.unpersist()

print("\nUnpersisted DataFrame to free up memory.")


Unpersisted DataFrame to free up memory.


In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# Sample Data
data = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 35),
    ("David", 15)
]

df = spark.createDataFrame(data, ["Name", "Age"])

df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [29]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"

In [30]:
age_category_udf = udf(categorize_age, StringType())

In [35]:
from pyspark.sql.types import StringType

spark.udf.register(
    "sql_categorize_age",
    categorize_age,
    StringType()
)

26/06/13 06:53:38 WARN SimpleFunctionRegistry: The function sql_categorize_age replaced a previously registered function.


<function __main__.categorize_age(age)>

In [37]:
df.createOrReplaceTempView("people")

In [38]:
# 4
sql_df = spark.sql("""
SELECT Name,
       Age,
       sql_categorize_age(Age) AS Category
FROM people
""")

sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+

